In [1]:
!git clone https://github.com/amanmandal7/COMP8410-A1_Evaluation_Presentation.git
%cd COMP8410-A1_Evaluation_Presentation/pair5_video_temporal

Cloning into 'COMP8410-A1_Evaluation_Presentation'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 86 (delta 16), reused 72 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 12.69 MiB | 24.65 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/COMP8410-A1_Evaluation_Presentation/pair5_video_temporal


In [2]:
!pip install -q transformers opencv-python

**Loading both the ViT deepfake detectors**

*   Model A - Older Baseline
*   Model B - Newer trining set  




In [3]:
from transformers import pipeline
import cv2
import numpy as np

model_a = pipeline("image-classification", model="dima806/deepfake_vs_real_image_detection")
model_b = pipeline("image-classification", model="prithivMLmods/Deep-Fake-Detector-v2-Model")

print("models loaded")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  343MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  343MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

models loaded


In [4]:
# checkig what labels each model outputs
print(model_a("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/hub/parrots.png"))
print(model_b("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/hub/parrots.png"))

[{'label': 'Real', 'score': 0.9503717422485352}, {'label': 'Fake', 'score': 0.04962829872965813}]
[{'label': 'Deepfake', 'score': 0.7101447582244873}, {'label': 'Realism', 'score': 0.2898552417755127}]


**Frame sampling and Per-frame scoring**

In [5]:
from PIL import Image

# pulling out about n frames evenly
def sample_frames(video_path, n_frames=12):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        print("Warning: couldn't read total frames correctly, defaulting to 1")
        total_frames = 1

    step_size = max(total_frames // n_frames, 1)

    frames_list = []
    curr_idx = 0

    while True:
        ret, current_frame = cap.read()
        if not ret:
            break

        if curr_idx % step_size == 0:
            # converting from BGR to RGB for the model
            rgb_frame = cv2.cvtColor(current_frame, cv2.COLOR_BGR2RGB)
            frames_list.append(Image.fromarray(rgb_frame))

        curr_idx += 1

    cap.release()
    return frames_list

# getting the probability that a frame is fake
def fake_prob(model_output):
    for item in model_output:
        lbl = item["label"].lower()
        if lbl == "fake" or lbl == "deepfake":
            return item["score"]

    for item in model_output:
        lbl = item["label"].lower()
        if lbl == "real" or lbl == "realism":
            return 1.0 - item["score"]

    print("Couldn't find expected labels in output:", model_output)
    return None

# running one model over all frames of one video
def score_video(model, frames):
    video_scores = []
    for f in frames:
        out = model(f)
        p = fake_prob(out)
        video_scores.append(p)
    return video_scores

**Quick test on the deepfake clip with model A**

In [6]:
print("loading test video...")
test_video_path = "data/celeb_synthesis.mp4"

import os
if not os.path.exists(test_video_path):
    print(f"Error: {test_video_path} not found!")
else:
    test_frames = sample_frames(test_video_path)
    print("frames sampled:", len(test_frames))

    # run scoring
    scores_result = score_video(model_a, test_frames)
    print("model A fake probs:", scores_result)

loading test video...
frames sampled: 13
model A fake probs: [0.00298297475092113, 0.0020875916816294193, 0.003449604846537113, 0.003861654084175825, 0.0015021474100649357, 0.0022713802754878998, 0.0013033044524490833, 0.0012071541277691722, 0.0012792868074029684, 0.0015857054386287928, 0.00287549733184278, 0.002533320104703307, 0.0016425888752564788]


**Running both models and the evaluation**

- Samples frames from all three videos
- Scores each frame’s fake probability with both detectors
- Per-video decision: mean probability + majority vote
- Temporal consistency: score variance and frame-to-frame change
- Compares decisions against each video’s true label

In [7]:
videos = {
    "real_video.mp4": "real",
    "real-1.mp4": "real",
    "celeb_synthesis.mp4": "fake",
}

# decide the whole video: fake if mean fake prob is above 0.5
def video_decision(scores):
    valid_scores = [s for s in scores if s is not None]

    if not valid_scores:
        print("Warning: no valid scores found for this video!")
        return 0.0, "unknown", "unknown"

    mean_score = np.mean(valid_scores)
    mean_label = "fake" if mean_score > 0.5 else "real"

    # doing majority vote logic
    fake_votes = 0
    for s in valid_scores:
        if s > 0.5:
            fake_votes += 1 # using += instead of + 1 cuz it's cleaner

    majority_label = "fake" if fake_votes > len(valid_scores) / 2 else "real"

    return mean_score, mean_label, majority_label

# temporal consistency: check how much the fake prob jumps around across consecutive frames
def temporal_stats(scores):
    valid_scores = [s for s in scores if s is not None]

    if len(valid_scores) < 2:
        return 0.0, 0.0

    variance = np.var(valid_scores)

    deltas = []
    for i in range(1, len(valid_scores)):
        diff = abs(valid_scores[i] - valid_scores[i-1])
        deltas.append(diff)

    avg_delta = np.mean(deltas) if deltas else 0.0

    return variance, avg_delta

# bundle the models into a list to loop through them easily
models_to_test = [
    ("Model A (dima806)", model_a),
    ("Model B (prithiv)", model_b)
]

# run everything across all models and videos
for model_name, current_model in models_to_test:
    print("=======", model_name, "======")

    for vid, truth in videos.items():
        vid_path = os.path.join("data", vid)

        # safety check for missing files
        if not os.path.exists(vid_path):
            print(f"Skipping {vid} - file not found in data/")
            continue

        frames_list = sample_frames(vid_path)

        # get scores for each frame
        scores = score_video(current_model, frames_list)

        # calculating metrics
        mean_score, mean_label, majority_label = video_decision(scores)
        variance, avg_delta = temporal_stats(scores)

        # print results
        print(vid, "| truth:", truth)
        print("  mean fake prob:", round(mean_score, 3),
              "| mean decision:", mean_label,
              "| majority:", majority_label)
        print("  temporal variance:", round(variance, 4),
              "| avg frame-to-frame change:", round(avg_delta, 4))

    print()

======= Model A (dima806) ======
real_video.mp4 | truth: real
  mean fake prob: 0.001 | mean decision: real | majority: real
  temporal variance: 0.0 | avg frame-to-frame change: 0.0001
real-1.mp4 | truth: real
  mean fake prob: 0.001 | mean decision: real | majority: real
  temporal variance: 0.0 | avg frame-to-frame change: 0.0001
celeb_synthesis.mp4 | truth: fake
  mean fake prob: 0.002 | mean decision: real | majority: real
  temporal variance: 0.0 | avg frame-to-frame change: 0.0008

======= Model B (prithiv) ======
real_video.mp4 | truth: real
  mean fake prob: 0.879 | mean decision: fake | majority: fake
  temporal variance: 0.0001 | avg frame-to-frame change: 0.0107
real-1.mp4 | truth: real
  mean fake prob: 0.851 | mean decision: fake | majority: fake
  temporal variance: 0.0013 | avg frame-to-frame change: 0.0401
celeb_synthesis.mp4 | truth: fake
  mean fake prob: 0.858 | mean decision: fake | majority: fake
  temporal variance: 0.0001 | avg frame-to-frame change: 0.0131

